# Feature selection
This notebook `01_feature_selection.ipynb` does feature selection of single-cell
profiles. It goes from single-cell profiles (post QC) to fully processed and
ready-to-analyse profiles by performing feature selection across all cell lines
and plates. This ensures that analyses are based only on features that were not
associated with confounders. For a more in-depth description of why and how this
is done, please refer to the section "Removing plate map effects" in the manuscript.

For a simpler introduction to feature selection with scmorph, see: https://scmorph.readthedocs.io/en/latest/tutorials/basics/feature_selection.html

In [1]:
import scmorph as sm
import pandas as pd
from plotnine import *
from scmorph.tl.hit_calling import *
from processing_helper import map_cell_lines
import pickle
import anndata as ad
from glob import glob
import pickle
import os

First, we will find all the anndata files created by `00_imageQC.ipynb` (or
downloaded from the CellPaintingGallery).

In [ ]:
INPUT_DIR = "workspace/profiles_assembled/profiles_qc/"
OUTPUT_DIR = "workspace/profiles_assembled/profiles_processed/"
adata_files = list(sorted(glob(f"{INPUT_DIR}/*_features.h5ad")))
adata_files

['workspace/profiles_assembled/profiles_qc/C2C12_R1_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/C2C12_R2_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/HEK293T_R1_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/HEK293T_R2_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/HepG2_R1_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/HepG2_R2_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/N2a_R1_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/N2a_R2_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/SHSY5Y_R1_features.h5ad',
 'workspace/profiles_assembled/profiles_qc/SHSY5Y_R2_features.h5ad']

In [ ]:
retained_features = {}
covariates = ["PlateID", "Row", "Column"]
for f in adata_files:
    print(f)
    adata = sm.read(f)
    map_cell_lines(adata)
    adata.obs["Row"] = adata.obs["Well"].str[0]
    adata.obs["Column"] = adata.obs["Well"].str[1:].astype(int)
    cell_line = adata.obs["CellLine"].iloc[0]
    replicate = adata.obs["Replicate"].iloc[0]
    sm.pp.remove_batch_effects(adata,
                               batch_key="PlateID",
                               treatment_key="treatment",
                               control="control_neg_mimic",
                               progress=False)
    for cov in covariates:
        sm.pp.kruskal_test(adata, test_column=cov, progress=True)
        retained_features[(cell_line, replicate, cov)] = sm.pp.kruskal_filter(adata, cov, copy=True).var.index
with open(f"{OUTPUT_DIR}/retained_features.pickle", "wb") as f:
    pickle.dump(retained_features, f)

In [4]:
features = []
for  (cell_line, replicate, cov), retained in retained_features.items():
    f = retained_features[(cell_line, replicate, cov)]
    f = f.to_frame().reset_index(drop=True)
    f.columns = ["feature"]
    f["CellLine"] = cell_line
    f["Replicate"] = replicate
    f["covariate"] = cov
    features.append(f)
features = pd.concat(features)
features = features.groupby("feature").size().loc[lambda x: x == len(adata_files)*len(covariates)].index

In [ ]:
adatas = []
for f in adata_files:
    adata = sm.read(f, backed="r")[:,features].to_memory()
    map_cell_lines(adata)
    sm.pp.remove_batch_effects(adata,
                               batch_key="PlateID",
                               treatment_key="treatment",
                               control="control_neg_mimic",
                               progress=False)
    adatas.append(adata)
adata = ad.concat(adatas)
adata.write(f"{OUTPUT_DIR}/all_adata_featFilt_batchCorr.h5ad")

/exports/igmm/eddie/khamseh-lab/jwagner/mamba/envs/micromorph/lib/python3.10/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


In [ ]:
adata_dict = {}
for cell_line in adata.obs["CellLine"].unique():
    adata_dict[cell_line] = []
    adata_cell_line = adata[adata.obs["CellLine"] == cell_line]
    adata_cell_line.write(f"{OUTPUT_DIR}/{cell_line}_features_processed.h5ad")
    for plate in adata_cell_line.obs["PlateID"].unique():
        adata_plate = adata_cell_line[adata_cell_line.obs["PlateID"] == plate]
        if len(adata_plate) > 0:
            adata_dict[cell_line].append(adata_plate)

In [ ]:
base_folder = os.path.basename(INPUT_DIR)
with open(f"{OUTPUT_DIR}/all_adata_featFilt_perPlate.pickle", "wb") as f:
    pickle.dump(adata_dict, f)